# Pilot v4 — Interaction-Aware Residual PromptIR
Mở notebook từ GitHub trên Colab rồi chọn GPU T4. Cell đầu tự tải mã nguồn từ GitHub; không cần upload thư mục `repo`. Code chạy trên SSD Colab; checkpoint và trạng thái lưu trên Drive.

FP32 là mốc kiểm tra ổn định. Bốn nhánh residual cùng tham số: `global_linear`, `regional_linear`, `regional_jordan`, `regional_lie`. Đây là thí nghiệm chưa có kết quả GPU. Endpoint chính là checkpoint **final / held-out composition**, không gọi toàn bộ lưới này là OOD.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
from pathlib import Path
import shutil, subprocess, sys, torch
REPO_URL = 'https://github.com/nthmaiii/all-in-one-image-restoration.git'
GIT_REF = 'main'  # Có thể thay bằng commit SHA để chạy lại đúng phiên bản.
CHECKOUT = Path('/content/interaction_v4_checkout')
WORK = CHECKOUT / 'repo'
RUN_ROOT = Path('/content/drive/MyDrive/nckh_coChau_runs/interaction_v4_fp32')
assert torch.cuda.is_available(), 'Chọn Runtime > Change runtime type > T4 GPU'
print(torch.__version__, torch.cuda.get_device_name(0))
if not CHECKOUT.exists():
    subprocess.run(['git', 'clone', '--depth', '1', REPO_URL, str(CHECKOUT)], check=True)
assert (CHECKOUT / '.git').is_dir(), f'{CHECKOUT} không phải Git checkout'
origin = subprocess.check_output(['git', '-C', str(CHECKOUT), 'remote', 'get-url', 'origin'], text=True).strip()
assert origin == REPO_URL, f'Git remote không khớp: {origin}'
subprocess.run(['git', '-C', str(CHECKOUT), 'fetch', '--depth', '1', 'origin', GIT_REF], check=True)
subprocess.run(['git', '-C', str(CHECKOUT), 'checkout', '--detach', 'FETCH_HEAD'], check=True)
assert (WORK / 'scripts/train_interaction_promptir.py').is_file(), 'Phiên bản Git chưa có pipeline v4'
print('Git commit:', subprocess.check_output(['git', '-C', str(CHECKOUT), 'rev-parse', 'HEAD'], text=True).strip())
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-r', str(WORK / 'requirements.txt')], check=True)
def run(*arguments):
    subprocess.run([sys.executable, *map(str, arguments)], cwd=WORK, check=True)


## Chuẩn bị dữ liệu thật
Cell này tải pretrained PromptIR, BSD68, Kodak24 và 200 ảnh sạch từ nguồn train. Nếu tải nguồn train thất bại, chương trình dừng; không thay bằng ảnh hoa văn ngẫu nhiên. Lưu dữ liệu chuẩn bị lên Drive để resume sau khi Colab reset.

In [ ]:
DATA_CACHE = Path('/content/drive/MyDrive/nckh_coChau_runs/interaction_v4_data')
if DATA_CACHE.exists():
    shutil.copytree(DATA_CACHE, WORK / 'data', dirs_exist_ok=True)
run('scripts/download_datasets.py', '--pilot_v2')
run('scripts/prepare_data_v2.py', '--require_real_images')
shutil.copytree(WORK / 'data', DATA_CACHE, dirs_exist_ok=True)


## Smoke test FP32 và AMP
Smoke test là kiểm tra thực thi, không phải bằng chứng chất lượng. Sau hai bước này cần kiểm tra ổn định dài hơn trước khi chạy toàn bộ 1.500 iterations. File smoke được lưu riêng.

In [ ]:
import json
for precision in ['fp32', 'amp']:
    archived = WORK / f'runs/smoke_interaction_v4_{precision}'
    if archived.exists():
        assert all(json.loads((archived / f'{v}_seed_42/status.json').read_text())['status'] == 'complete' for v in ['baseline', 'regional_lie'])
        print('Smoke already complete:', precision)
        continue
    for variant in ['baseline', 'regional_lie']:
        # Smoke root được script cố định; tách theo precision bằng thư mục làm việc lưu trữ sau lượt chạy.
        directory = WORK / f'runs/smoke_interaction_v4/{variant}_seed_42'
        status_path = directory / 'status.json'
        if status_path.exists():
            protocol = json.loads((directory / 'protocol.json').read_text())
            assert protocol['shared']['precision'] == precision, 'Smoke root chứa precision khác; kiểm tra lại lượt chạy'
            if json.loads(status_path.read_text())['status'] == 'complete':
                continue
        command = ['scripts/train_interaction_promptir.py', '--variant', variant, '--smoke_test', '--precision', precision]
        if (directory / 'latest_checkpoint.pth').exists():
            command += ['--resume']
        run(*command)
    smoke_root = WORK / 'runs/smoke_interaction_v4'
    smoke_root.rename(archived)


## Kiểm tra ổn định 100 iterations
Dùng một thư mục kết quả riêng và một tập test nhỏ. Chạy baseline trước cho mỗi seed. Resume được tự động dùng nếu đã có checkpoint; mọi thay đổi code/dữ liệu/lịch học sẽ bị phát hiện qua protocol. AMP chưa ổn định thì giữ FP32 và xem status.json để tìm vị trí lỗi.

In [ ]:
def experiment(variant, seed, output, iters=1500, test_images=30):
    directory = output / f'{variant}_seed_{seed}'
    import json
    status_path = directory / 'status.json'
    if status_path.exists() and json.loads(status_path.read_text())['status'] == 'complete':
        print('Already complete:', directory)
        return
    command = ['scripts/train_interaction_promptir.py', '--variant', variant, '--seed', seed, '--iters', iters, '--precision', 'fp32', '--test_images', test_images, '--output_dir', output]
    if (directory / 'latest_checkpoint.pth').exists():
        command += ['--resume']
    run(*command)
STABILITY_ROOT = RUN_ROOT.parent / 'interaction_v4_stability100'
for variant in ['baseline', 'regional_lie']:
    for seed in [42, 123, 2024]:
        experiment(variant, seed, STABILITY_ROOT, iters=100, test_images=2)


## Pilot chính — baseline trước, sau đó các đối chứng
Lưới composition: 30 ảnh × 6 tổ hợp × 2 thứ tự = 360 mẫu. Lưới extrapolation: 30 × 4 × 2 = 240 mẫu. Mỗi checkpoint được đo cả khi bật/tắt residual. Đánh giá này lớn hơn v3 nên thời gian chạy cần đo thực tế. Nếu bị ngắt, giữ nguyên cấu hình và chạy lại cell setup + dữ liệu + cell dưới để resume.

In [ ]:
VARIANTS = ['baseline', 'global_linear', 'regional_linear', 'regional_jordan', 'regional_lie']
SEEDS = [42, 123, 2024]
for variant in VARIANTS:
    for seed in SEEDS:
        experiment(variant, seed, RUN_ROOT)


In [ ]:
run('scripts/compare_interaction_runs.py', '--output_dir', RUN_ROOT)
print('Gửi comparison.json, protocol_seed_*.json và các summary.json/status.json để phân tích kết quả.')
